# Preprocesamiento

Apartado del enunciado: **3.1**, mínimo tres técnicas de preprocesamiento, indicando qué problemática
resuelve cada una.

## Objetivos

1. Saber **cómo afecta cada preprocesado a los modelos**, con un ajuste mínimo de hiperparámetros (el
   ajuste completo se hace en el modelado).
2. Saber **si imputar los nulos ayuda o no a Random Forest**, que puede trabajar con NaN directamente.

## Estructura del cuaderno

El cuaderno está pensado para que **añadir un preprocesado nuevo sea solo añadir una línea**:

1. **Configuración**: librerías, semilla y umbrales (todos salen de `01_eda.ipynb`).
2. **Carga del dataset** y resumen de nulos.
3. **Catálogo de pasos**: cada paso es una función `df -> df` que imprime lo que ha hecho.
   Están agrupados en: eliminar filas, valores imposibles a NaN, incoherencias cruzadas, columnas y pasos
   opcionales.
4. **Registro de preprocesados** (`PREPROCESADOS`): cada preprocesado es una lista de pasos.
5. **Codificación, imputación, escalado y modelos**: se hacen dentro de un `Pipeline`, que se ajusta solo
   con el train de cada partición para evitar fugas de información.
6. **Evaluación**: todos los preprocesados se prueban con los mismos modelos y las mismas particiones.
7. **Resultados**, incluido el experimento de imputación.

| ID | Pasos que añade | Problemática del EDA que resuelve |
| :--- | :--- | :--- |
| `P0_base` | Ninguno (solo codificación y mediana del pipeline) | Ninguna. Es la base de comparación |
| `P1_limpieza` | Limpieza común | Menores, valores imposibles, incoherencias entre variables, ejercicio con decimales, `colesterol_total` corrupto |
| `P2_diabetes` | P1 + recodificar `diabetes_mellitus` | 597 diabéticos por criterio analítico sin marcar |
| `P3_pcr` | P2 + eliminar PCR > 10 mg/L | PCR de proceso agudo, que no refleja riesgo crónico |
| `P4_discretizacion` | P3 + sustituir `edad` por su grupo etario e `indice_masa_corporal` por su categoría de la OMS | El EDA propone usar grupos etarios clínicos y discretizar el IMC |
| `P5_smote` | P4 + SMOTE dentro del `Pipeline` | Desbalanceo de clases |
| `P6_adasyn` | P4 + ADASYN dentro del `Pipeline` | Desbalanceo de clases, con más muestras donde el límite entre clases es más difícil |
| `P7_rfe` | P4 + selección de variables con RFE dentro del `Pipeline` | Variables poco informativas o redundantes |

## 1. Configuración

Se fija la **semilla** y se reúnen los **umbrales de dominio** en una sola celda. Si se cambia un umbral,
se cambia aquí y afecta a todos los preprocesados.

In [1]:
from pathlib import Path
import sklearn
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.svm import LinearSVC

In [2]:
pd.set_option("display.max_columns", None, "display.width", 200)

SEED = 123
TARGET = "riesgo_cardiovascular"

# Ruta al dataset: funciona desde notebooks/, desde notebooks/workspace/ y desde la raíz del proyecto
RUTAS = ["../data/raw/enfermedad_cardiovascular.csv",
         "../../data/raw/enfermedad_cardiovascular.csv",
         "data/raw/enfermedad_cardiovascular.csv"]
DATA_PATH = next(Path(r) for r in RUTAS if Path(r).exists())

# --- Umbrales del EDA (01_eda.ipynb) ---
EDAD_MINIMA = 18           # el enunciado limita la población a adultos
IMC_MINIMO = 13            # por debajo de 12-13 es biológicamente imposible (mínimo en los datos: 3.6)
EJERCICIO_MAXIMO = 7       # días por semana
HBA1C_MINIMA = 3           # HbA1c < 3 % no es fisiológica
SISTOLICA_MINIMA = 60      # mmHg, ver 01_eda: "Mínimos plausibles de la presión arterial"
DIASTOLICA_MINIMA = 40     # mmHg, ver 01_eda: "Mínimos plausibles de la presión arterial"
PCR_MAXIMA = 10            # mg/L; por encima indica un proceso agudo, no riesgo crónico
HBA1C_DIABETES = 6.5       # % (criterio diagnóstico de diabetes)
GLUCOSA_DIABETES = 126     # mg/dL en ayunas (criterio diagnóstico de diabetes)

# --- Variables categóricas ---
NOMINALES = ["tipo_alimentacion", "antecedentes_familiares_tipo"]
ORDINAL = ["nivel_actividad"]
ORDEN_ACTIVIDAD = ["Sedentario", "Ligeramente activo", "Moderadamente activo", "Muy activo"]
# --- Variables booleanas ---
BOOLEANOS = ['fumador_actual', 'hipertension_previa', 'diabetes_mellitus', 'tratamiento_estatinas', 'uso_antihipertensivos']
# --- Variables numéricas ---
NUMERICAS = ['edad', 'presion_sistolica', 'presion_diastolica', 'colesterol_total',  'colesterol_hdl', 'colesterol_ldl', 'trigliceridos', 'glucosa_ayunas', 'indice_masa_corporal', 'frecuencia_cardiaca_reposo', 'horas_sueno_diarias', 'consumo_alcohol_semanal', 'anos_fumador', 'proteina_c_reactiva', 'acido_urico', 'clearance_creatinina', 'frecuencia_ejercicio_semanal', 'perimetro_cintura', 'hemoglobina_glicosilada']

## 2. Carga del dataset

El CSV se lee sin modificarlo. `resumen_nulos` se reutiliza después para ver cuántos NaN deja cada
preprocesado.

In [3]:
def resumen_nulos(df):
    print("Valores nulos:", df.isnull().sum().sum())
    resumen = pd.DataFrame({
        "Nulos": df.isnull().sum(),
        "Porcentaje (%)": (df.isnull().mean() * 100).round(2),
    })
    return resumen[resumen["Nulos"] > 0].sort_values(by="Nulos", ascending=False)


df = pd.read_csv(DATA_PATH)
print(df.shape)
resumen_nulos(df)

(2254, 28)
Valores nulos: 584


,Nulos,Porcentaje (%)
horas_sueno_diarias,180,7.99
proteina_c_reactiva,157,6.97
colesterol_ldl,135,5.99
glucosa_ayunas,112,4.97


## 3. Catálogo de pasos

Cada paso recibe un `DataFrame` y devuelve otro, sin modificar el original. Todos usan dos funciones
auxiliares para que el mensaje que imprimen tenga siempre el mismo formato.

Son **reglas fijas** que no aprenden nada de los datos (no calculan medias ni medianas), así que se pueden
aplicar antes de partir en train y test sin provocar fugas de información.

In [4]:
def eliminar_filas(df, mascara, motivo):
    antes = len(df)
    df = df[~mascara]
    print(f"{antes - len(df)} filas eliminadas por {motivo}. Hay {len(df)} filas restantes.")
    return df


def poner_nan(df, mascara, columnas, motivo):
    df = df.copy()
    df.loc[mascara, columnas] = np.nan
    print(f"{int(mascara.sum())} valores con {motivo} se han puesto a NaN en {', '.join(columnas)}.")
    return df

### 3.1 Eliminar filas

- **Menores de edad** (4): el enunciado limita la población a adultos.
- **Más años fumando que edad** (1): el registro entero es incoherente.

In [5]:
def eliminar_menores(df):
    return eliminar_filas(df, df["edad"] < EDAD_MINIMA, "ser menores de edad")


def eliminar_fumador_incoherente(df):
    return eliminar_filas(df, df["anos_fumador"] > df["edad"], "tener más años fumando que edad")

### 3.2 Valores imposibles a NaN

Un valor imposible es un error de registro, pero el resto del paciente puede ser válido. Por eso la celda
pasa a **NaN** y la fila se conserva; después la imputa el pipeline (sección 5).

Las reglas están en una tabla (`REGLAS_RANGO`): para añadir una nueva basta con añadir una línea.
Los mínimos de la presión arterial están justificados con referencias clínicas en `01_eda.ipynb`.

**La presión invertida (diastólica >= sistólica) se comprueba antes que los mínimos.** Si fuera al revés,
un paciente con sistólica 52,2 y diastólica 90,6 perdería primero la sistólica, y la comparación
`90,6 >= NaN` daría `False`: su diastólica, probablemente errónea, se conservaría. Comprobándola antes,
las dos presiones pasan a NaN, igual que en el resto de casos invertidos.

In [6]:
def nan_presion_invertida(df):
    return poner_nan(df, df["presion_diastolica"] >= df["presion_sistolica"],
                     ["presion_sistolica", "presion_diastolica"], "presión diastólica >= sistólica")


# (columna, condición que hace imposible el valor, descripción)
REGLAS_RANGO = [
    ("colesterol_hdl", lambda s: s <= 0, "colesterol_hdl negativo o 0"),
    ("trigliceridos", lambda s: s <= 0, "trigliceridos negativos o 0"),
    ("consumo_alcohol_semanal", lambda s: s < 0, "consumo de alcohol semanal negativo"),
    ("indice_masa_corporal", lambda s: s < IMC_MINIMO, f"IMC menor que {IMC_MINIMO}"),
    ("frecuencia_ejercicio_semanal", lambda s: s > EJERCICIO_MAXIMO, f"más de {EJERCICIO_MAXIMO} días de ejercicio semanal"),
    ("hemoglobina_glicosilada", lambda s: s < HBA1C_MINIMA, f"hemoglobina glicosilada menor que {HBA1C_MINIMA}%"),
    ("presion_sistolica", lambda s: s < SISTOLICA_MINIMA, f"presión sistólica menor que {SISTOLICA_MINIMA} mmHg"),
    ("presion_diastolica", lambda s: s < DIASTOLICA_MINIMA, f"presión diastólica menor que {DIASTOLICA_MINIMA} mmHg"),
]

def nan_valores_imposibles(df):
    for columna, condicion, motivo in REGLAS_RANGO:
        df = poner_nan(df, condicion(df[columna]), [columna], motivo)
    return df

### 3.3 Incoherencias entre variables a NaN

Cuando dos variables se contradicen no se sabe cuál de las dos está mal, así que **ambas** pasan a NaN.
La presión invertida es también una incoherencia de este tipo, pero se trata antes, en la sección 3.2.

| Incoherencia | Por qué es un error |
| :--- | :--- |
| IMC > 35 y cintura < 70 cm | Obesidad grado II con cintura de persona delgada |
| IMC < 18,5 y cintura > 110 cm | Bajo peso con cintura de obesidad abdominal |
| HbA1c < 5,5 % y glucosa > 180 mg/dL | HbA1c normal con glucosa de diabetes descompensada |
| HbA1c > 9 % y glucosa < 80 mg/dL | HbA1c de diabetes mal controlada con glucosa normal-baja |

In [7]:
# (condición sobre el DataFrame, columnas que pasan a NaN, descripción)
REGLAS_CRUZADAS = [
    (lambda d: (d["indice_masa_corporal"] > 35) & (d["perimetro_cintura"] < 70),
     ["indice_masa_corporal", "perimetro_cintura"], "IMC mayor que 35 y cintura menor que 70"),
    (lambda d: (d["indice_masa_corporal"] < 18.5) & (d["perimetro_cintura"] > 110),
     ["indice_masa_corporal", "perimetro_cintura"], "IMC menor que 18.5 y cintura mayor que 110"),
    (lambda d: (d["hemoglobina_glicosilada"] < 5.5) & (d["glucosa_ayunas"] > 180),
     ["hemoglobina_glicosilada", "glucosa_ayunas"], "hemoglobina glicosilada menor que 5.5% y glucosa en ayunas mayor que 180"),
    (lambda d: (d["hemoglobina_glicosilada"] > 9) & (d["glucosa_ayunas"] < 80),
     ["hemoglobina_glicosilada", "glucosa_ayunas"], "hemoglobina glicosilada mayor que 9% y glucosa en ayunas menor que 80"),
]


def nan_incoherencias_cruzadas(df):
    for condicion, columnas, motivo in REGLAS_CRUZADAS:
        df = poner_nan(df, condicion(df), columnas, motivo)
    return df

### 3.4 Columnas

**Las variables que cuentan unidades enteras se redondean.** El EDA detectó decimales en variables que
deberían ser enteras:

| Variable | Con decimales | Redondeo | Motivo |
| :--- | :--- | :--- | :--- |
| `edad` | 2.042 | Hacia abajo (45,8 → 45) | La edad son años cumplidos |
| `frecuencia_ejercicio_semanal` | 81,5 % | Hacia arriba (6,1 → 7) | Si ha hecho ejercicio algún rato de un día, ese día cuenta |
| `anos_fumador` | 1.588 | Al más cercano | Años de tabaquismo |
| `frecuencia_cardiaca_reposo` | 2.014 | Al más cercano | Latidos por minuto |
| `consumo_alcohol_semanal` | 2.055 | Al más cercano | Unidades de alcohol por semana |

- El redondeo se hace después de eliminar a los menores y de poner a NaN los imposibles. Como la edad se
  redondea hacia abajo, nadie que tuviera 18 años o más baja de 18, y tampoco aparece ningún paciente con
  más años fumando que edad.
- El ejercicio se redondea después de poner a NaN los valores mayores que 7 (sección 3.2), así que el
  resultado nunca pasa de 7. Su media sube alrededor de medio día.
- En el resto, el redondeo no sesga la variable: la media y la correlación con el target apenas cambian
  (menos de 0,01). Los fumadores actuales con 0 años pasan de 123 a 132, porque quien lleva menos de
  medio año fumando queda en 0 (el EDA ya consideraba justificable este caso).

**`colesterol_total` se elimina**: el 24,8 % de los pacientes tiene un total menor que HDL + LDL (imposible),
el 57,76 % no cuadra con la fórmula HDL + LDL + TG/5 y su correlación con el target es muy baja (ver EDA).
Recalcularla con la fórmula no aportaría información nueva, porque sería una combinación lineal de
variables que el modelo ya recibe.

In [8]:
# columna: función de redondeo
REDONDEO = {
    "edad": np.floor,
    "frecuencia_ejercicio_semanal": np.ceil,
    "anos_fumador": np.round,
    "frecuencia_cardiaca_reposo": np.round,
    "consumo_alcohol_semanal": np.round,
}


def redondear_enteros(df):
    df = df.copy()
    for columna, redondeo in REDONDEO.items():
        cambiados = int((df[columna] != redondeo(df[columna])).sum() - df[columna].isna().sum())
        df[columna] = redondeo(df[columna])
        print(f"{cambiados} valores de {columna} se han redondeado ({redondeo.__name__}).")
    return df


def eliminar_colesterol_total(df):
    print("Columna colesterol_total eliminada.")
    return df.drop(columns="colesterol_total")

### 3.5 Pasos opcionales

- **Recodificar diabetes** (muy recomendable): según los criterios diagnósticos, un paciente con
  HbA1c >= 6,5 % o glucosa en ayunas >= 126 mg/dL es diabético. El EDA encontró 597 pacientes que los
  cumplen y no están marcados en `diabetes_mellitus`.
- **Eliminar PCR > 10 mg/L**: una PCR tan alta indica un proceso agudo (infección, inflamación), no riesgo
  cardiovascular crónico, así que se supone que la prueba no es válida. Los pacientes **sin** PCR medida
  (NaN) se conservan. Al eliminar filas, el conjunto de test de este preprocesado es algo menor que el de
  los demás (sección 6).

In [9]:
def recodificar_diabetes(df):
    df = df.copy()
    cond = (df["hemoglobina_glicosilada"] >= HBA1C_DIABETES) | (df["glucosa_ayunas"] >= GLUCOSA_DIABETES)
    nuevos = int((cond & (df["diabetes_mellitus"] == 0)).sum())
    df.loc[cond, "diabetes_mellitus"] = 1
    print(f"{int(cond.sum())} pacientes con hemoglobina glicosilada >= {HBA1C_DIABETES}% o glucosa en ayunas "
          f">= {GLUCOSA_DIABETES} mg/dL se han puesto como diabéticos ({nuevos} no lo estaban).")
    return df


def eliminar_pcr_alta(df):
    return eliminar_filas(df, df["proteina_c_reactiva"] > PCR_MAXIMA,
                          f"tener proteína C reactiva mayor que {PCR_MAXIMA} mg/L")

- **Discretizar edad e IMC** (P4): el EDA propone sustituir la edad por **grupos etarios clínicos** y discretizar el IMC según la
  clasificación de la **OMS**. La relación de la edad con el riesgo es progresiva: el riesgo alto crece de un grupo al siguiente.

  | Variable | Categorías (código) |
  | :--- | :--- |
  | `edad` | 0: 18-39, 1: 40-54, 2: 55-64, 3: >= 65 años |
  | `indice_masa_corporal` | 0: bajo peso (< 18,5), 1: normal (18,5-25), 2: sobrepeso (25-30), 3: obesidad I (30-35), 4: obesidad II (35-40), 5: obesidad III (>= 40) |

  Las dos columnas **conservan su nombre** pero pasan a contener el **código ordinal** de la categoría (las categorías tienen orden). Así `NUMERICAS` y el
  `Pipeline` de la sección 5 no cambian: la mediana de un código es siempre un código válido (no aparece un 2,5 sin sentido) y el escalado se aplica igual que al
  resto de variables numéricas. Los NaN del IMC se mantienen como NaN y los imputa el `Pipeline`. Los cortes son reglas fijas, así que se pueden aplicar antes de partir en
  train y test.

In [10]:
EDAD_CORTES = [18, 40, 55, 65, 120]                 # grupos etarios clínicos del EDA, intervalos [a, b)
IMC_CORTES = [0, 18.5, 25, 30, 35, 40, np.inf]      # clasificación de la OMS, intervalos [a, b)


def discretizar_edad_imc(df):
    df = df.copy()
    df["edad"] = pd.cut(df["edad"], bins=EDAD_CORTES, labels=False, right=False)
    df["indice_masa_corporal"] = pd.cut(df["indice_masa_corporal"], bins=IMC_CORTES, labels=False, right=False)
    print(f"edad e indice_masa_corporal se han sustituido por su código de categoría "
          f"({df['edad'].nunique()} grupos etarios y {df['indice_masa_corporal'].nunique()} categorías de IMC; "
          f"{int(df['indice_masa_corporal'].isna().sum())} NaN de IMC se mantienen).")
    return df

## 4. Registro de preprocesados

Cada preprocesado es una **lista de pasos** que se aplican en orden. Cada uno añade **un solo cambio** al
anterior, así que la diferencia de métricas entre dos preprocesados consecutivos se puede atribuir a ese
cambio.

### Cómo añadir un preprocesado nuevo

1. Si hace falta, escribir el paso en la sección 3 como una función `df -> df`
   (o añadir una línea a `REGLAS_RANGO` / `REGLAS_CRUZADAS`).
2. Añadir una entrada a `PREPROCESADOS`, por ejemplo:

```python
def mi_paso(df):
    ...
    return df

PREPROCESADOS["P4_mi_paso"] = PREPROCESADOS["P3_pcr"] + [mi_paso]
```

3. Si el cambio está en la codificación, la imputación o el escalado, va en `construir_pipeline`
   (sección 5).

El resto del cuaderno (evaluación y resultados) recorre `PREPROCESADOS` y no hay que tocarlo.

In [10]:
LIMPIEZA_COMUN = [
    eliminar_menores,              # Eliminar menores de edad (ver sección 3.1)
    eliminar_fumador_incoherente,  # Eliminar fumadores con años de fumador > edad (ver sección 3.1)
    nan_presion_invertida,         # Eliminar presiones invertidas antes que los mínimos de presión (ver sección 3.2)
    nan_valores_imposibles,        # Eliminar valores científicamente imposibles (ver sección 3.3)
    redondear_enteros,             # Realizar redondeos de columnas con valores decimales a enteros después de poner a NaN los imposibles (ver sección 3.4)
    eliminar_colesterol_total,     # Eliminar la columna de colesterol total (ver sección 3.5)
    nan_incoherencias_cruzadas,    # Eliminar incoherencias entre los valores de diferentes columnas (ver sección 3.6)
]

PREPROCESADOS = {
    "P0_base": [],
    "P1_limpieza": LIMPIEZA_COMUN,
    "P2_diabetes": LIMPIEZA_COMUN + [recodificar_diabetes], # Recodificar diabetes según HbA1c y glucosa en ayunas para indicar casos de diabetes que no estaban claramente definidos (ver sección 3.7)
    "P3_pcr": LIMPIEZA_COMUN + [recodificar_diabetes, eliminar_pcr_alta], # Eliminar pacientes con PCR > 10 mg/L (ver sección 3.8)
    "P4_discretizacion": LIMPIEZA_COMUN + [recodificar_diabetes, eliminar_pcr_alta, discretizar_edad_imc] # Sustituir edad e IMC por su categoría (ver sección 3.5)
}


def aplicar_preprocesado(df, pasos):
    for paso in pasos:
        df = paso(df)
    return df

**Preprocesados que aprenden de los datos (P5, P6 y P7).** El sobremuestreo (SMOTE, ADASYN) y la selección de variables (RFE) no son pasos `df -> df`:
calculan vecinos o coeficientes a partir de los datos, así que si se aplicaran antes de partir en train y test provocarían fugas de información. Por eso se registran aparte,
en `PREPROCESADOS_PIPELINE`, y se insertan **dentro del `Pipeline`** de la sección 5, entre el preprocesado y el modelo, donde se ajustan solo con el train de cada partición.
Todos parten de los datos de **P4** (`base`), el último preprocesado incremental de arriba.

- **SMOTE** (`P5_smote`): genera pacientes sintéticos de las clases minoritarias interpolando entre cada paciente y sus vecinos más cercanos de su misma clase.
- **ADASYN** (`P6_adasyn`): igual que SMOTE, pero genera más muestras donde la clase minoritaria está más mezclada con las otras, que son las zonas más difíciles.
- **RFE** (`P7_rfe`): entrena un modelo, elimina la variable menos importante y repite. Se usa `RFECV` (3 particiones, `f1_macro`, mínimo 5 variables), que además elige cuántas conservar, con
  una regresión logística como estimador (sus coeficientes miden la importancia de cada variable ya escalada). El `RFECV` normal se queda con el número de variables de **mejor puntuación**, y en estos datos la
  puntuación sube con cada variable que se añade, así que conservaría las 30 y el preprocesado no haría nada. Por eso se aplica la **regla del error estándar**: se conserva el **menor número de variables
  cuya puntuación media queda a menos de un error estándar de la mejor** (error estándar = desviación entre particiones / raíz del número de particiones). Es una regla habitual para preferir el modelo más
  sencillo cuando la diferencia con el mejor está dentro del ruido. La clase `RFECVUnErrorEstandar` solo cambia ese criterio de elección y deja el resto de `RFECV` igual.

Para añadir otro preprocesado de este tipo basta con añadir una entrada a `PREPROCESADOS_PIPELINE`.

In [12]:
from imblearn.over_sampling import ADASYN, SMOTE
from sklearn.feature_selection import RFE, RFECV


class RFECVUnErrorEstandar(RFECV):
    # RFECV que elige el menor número de variables con puntuación a menos de un error estándar de la mejor (requiere step=1)
    def fit(self, X, y, **params):
        super().fit(X, y, **params)
        media = np.asarray(self.cv_results_["mean_test_score"])
        n_particiones = sum(k.startswith("split") and k.endswith("_test_score") for k in self.cv_results_)
        error = np.asarray(self.cv_results_["std_test_score"]) / np.sqrt(n_particiones)
        mejor = media.argmax()
        n_elegido = self.min_features_to_select + int(np.where(media >= media[mejor] - error[mejor])[0].min())
        rfe = RFE(self.estimator, n_features_to_select=n_elegido, step=self.step).fit(X, y)
        self.n_features_, self.support_, self.ranking_, self.estimator_ = rfe.n_features_, rfe.support_, rfe.ranking_, rfe.estimator_
        return self


# nombre: {preprocesado de datos del que parte, sobremuestreador (o None), selección de variables (o None)}
PREPROCESADOS_PIPELINE = {
    "P5_smote": {"base": "P4_discretizacion", "sampler": SMOTE(random_state=SEED), "seleccion": None},
    "P6_adasyn": {"base": "P4_discretizacion", "sampler": ADASYN(random_state=SEED), "seleccion": None},
    "P7_rfe": {"base": "P4_discretizacion", "sampler": None,
               "seleccion": RFECVUnErrorEstandar(estimator=LogisticRegression(solver='lbfgs', max_iter=1000, random_state=SEED),
                                  step=1, cv=3, scoring="f1_macro", min_features_to_select=5)},
}

Se aplica cada preprocesado al dataset completo y se guarda el resultado en `DATOS`. Los mensajes
muestran cuántos valores cambia cada paso.

In [13]:
DATOS = {}
for nombre, pasos in PREPROCESADOS.items():
    print(f"\n----------- {nombre} -----------")
    DATOS[nombre] = aplicar_preprocesado(df, pasos)
    print("Dimensiones:", DATOS[nombre].shape)


----------- P0_base -----------
Dimensiones: (2254, 28)

----------- P1_limpieza -----------
4 filas eliminadas por ser menores de edad. Hay 2250 filas restantes.
0 filas eliminadas por tener más años fumando que edad. Hay 2250 filas restantes.
44 valores con presión diastólica >= sistólica se han puesto a NaN en presion_sistolica, presion_diastolica.
4 valores con colesterol_hdl negativo o 0 se han puesto a NaN en colesterol_hdl.
4 valores con trigliceridos negativos o 0 se han puesto a NaN en trigliceridos.
177 valores con consumo de alcohol semanal negativo se han puesto a NaN en consumo_alcohol_semanal.
19 valores con IMC menor que 13 se han puesto a NaN en indice_masa_corporal.
41 valores con más de 7 días de ejercicio semanal se han puesto a NaN en frecuencia_ejercicio_semanal.
1 valores con hemoglobina glicosilada menor que 3% se han puesto a NaN en hemoglobina_glicosilada.
0 valores con presión sistólica menor que 60 mmHg se han puesto a NaN en presion_sistolica.
3 valores con

**Nulos tras la limpieza común.** Todos estos NaN los imputa el pipeline (sección 5).

In [12]:
resumen_nulos(DATOS["P1_limpieza"])

Valores nulos: 990


,Nulos,Porcentaje (%)
horas_sueno_diarias,180,8.00
consumo_alcohol_semanal,177,7.87
proteina_c_reactiva,157,6.98
colesterol_ldl,134,5.96
glucosa_ayunas,113,5.02
indice_masa_corporal,53,2.36
presion_diastolica,47,2.09
presion_sistolica,44,1.96
frecuencia_ejercicio_semanal,41,1.82
perimetro_cintura,34,1.51


No hay ninguna columna nominal ni booleana que tenga nulos, tanto en P0 como en P1, por lo que no se tiene que aplicar técnicas como imputar los nulos con la moda antes de realizar transformaciones de las variables. Aunque el número de nulos ha crecido, no hay ninguna columna especialmente afectada por la falta de valores para decidir eliminarla del conjunto de datos por pérdida de información relevante. 

In [13]:
def detectar_baja_varianza(df, freq_ratio_umbral=19, unique_pct_umbral=10):
    resultados = []
    columnas_a_evaluar = df.columns
    columnas_a_evaluar = columnas_a_evaluar.drop('riesgo_cardiovascular')
    for col in columnas_a_evaluar:
        conteo_valores = df[col].value_counts(dropna=True)
        if len(conteo_valores) == 0:
            continue
        # Porcentaje de valores únicos sobre el total de objetos (para comparar con unique_pct_umbral)
        n_unicos = len(conteo_valores)
        n_filas = len(df[col].dropna())
        pct_unicos = (n_unicos / n_filas) * 100
        # Ratio de frecuencia entre el valor más frecuente y el segundo más frecuente (para comparar con freq_ratio_umbral)
        if n_unicos >= 2:
            freq_1 = conteo_valores.iloc[0]
            freq_2 = conteo_valores.iloc[1]
            freq_ratio = freq_1 / freq_2 if freq_2 > 0 else np.inf
        else:
            # Si solo hay 1 valor único (columna 100% constante)
            freq_ratio = np.inf
            
        resultados.append({
            'variable': col,
            'n_unicos': n_unicos,
            'porcentaje_unicos': pct_unicos,
            'ratio_frecuencia': freq_ratio,
            'valor_mas_frecuente': conteo_valores.index[0],
            'freq_valor_mas_frecuente': conteo_valores.iloc[0]
        })
        
    df_metricas = pd.DataFrame(resultados)
    
    # Si el ratio de frecuencia supera un límite establecido y el porcentaje de valores únicos cae por debajo de un límite establecido,
    # podemos considerar la variable como una variable con varianza cercana a cero.
    df_baja_var = df_metricas[
        (df_metricas['ratio_frecuencia'] > freq_ratio_umbral) & 
        (df_metricas['porcentaje_unicos'] < unique_pct_umbral)
    ].reset_index(drop=True)
    
    return df_metricas, df_baja_var


metricas_completas, columnas_baja_varianza = detectar_baja_varianza(DATOS["P1_limpieza"])
print("Métricas de varianza por variable:")
print(metricas_completas)
print("\nVariables con varianza cercana a cero (posibles candidatas a eliminar):")
print(columnas_baja_varianza)

Métricas de varianza por variable:
                        variable  n_unicos  porcentaje_unicos  ratio_frecuencia valor_mas_frecuente  freq_valor_mas_frecuente
0                           edad        83           3.688889          1.085714                59.0                        76
1              presion_sistolica       732          33.182230          1.113208              196.95                        59
2             presion_diastolica       558          25.329097          1.000000                80.0                        12
3                 colesterol_hdl       652          29.029386          1.076923                46.3                        14
4                 colesterol_ldl      1105          52.221172          1.000000               125.0                         7
5                  trigliceridos      1344          59.839715          1.500000               452.7                        54
6                 glucosa_ayunas       871          40.758072          1.000000    

Aunque claramente la varianza ha cambiado respecto a la ejecución del EDA, especialmente en variables que se han redondeado, que claramente ha provocado la reducción de sus valores únicos, no han aparecido ninguna columna con una varianza cercana a cero para considerar eliminarla.

## 5. Evaluación

La codificación, imputación, escalado y modelos se hacen dentro de un `Pipeline`, que se **ajusta solo con el train** de cada partición. Primero hay que determinar transformaciones categóricas que serán comunes a todos los pipelines del preprocesado, que es el tratamiento de las variables nominales y ordinales, puesto que los modelos no procesan cadenas de texto directamente.

- **Dummies** (`OneHotEncoder`, `drop="first"`) para `tipo_alimentacion` y `antecedentes_familiares_tipo`, ya que se tratan de variables sin jerarquía.
- **Ordinal 0-3** (`OrdinalEncoder`) para `nivel_actividad` porque sus categorías tienen un orden natural.

In [14]:
# Común a todos los pipelines de preprocesado
transformadores_categoricos = [
    ("nominales", OneHotEncoder(drop="first", handle_unknown="ignore"), NOMINALES),
    ("ordinal", OrdinalEncoder(categories=[ORDEN_ACTIVIDAD]), ORDINAL),
    ("booleanos", "passthrough", BOOLEANOS)
]

En cuanto a los modelos seleccionados para la práctica se deciden 3 modelos base para más adelante seleccionar 2 de ellos:

* Logistic Regression
* LinearSVC
* SVM

Y 3 ensambles:

* POR DETERMINAR
* POR DETERMINAR
* POR DETERMINAR

Como se puede observar, algunos de estos modelos son altamente sensibles a la escala numérica y dependen de un escalado estricto (como regresión logística) y no procesan valores nulos de forma nativa. Por lo tanto, para este primer experimento que plantea determinar qué base de preprocesamiento en relación con la limpieza de datos (`P0`, `P1`, `P2`, `P3`) resulta más eficiente, se realizará un pipeline que, además de realizar las transformaciones categóricas, impute valores por la mediana y realice un normalización con un escalado estándar. Como únicamente hay presencia de nulos en variables numerales, no es necesaria una estrategia de imputación como, por ejemplo, la moda en variables nominales o booleanas.

Es importante mencionar que tanto la mediana como el escalado se deben calcular únicamente con los valores del conjunto de entrenamiento. Al encapsular las transformaciones dentro de este pipeline y usando `pipeline_fold.fit(X_train_fold, y_train_fold)` en la función de evaluación (se verá más adelante) se garantiza que se aplica únicamente sobre los datos de entrenamiento y el conjunto de test haga las predicciones en el `predict()` con dichas transformaciones del train.

In [15]:
# Preprocesado que imputa valores nulos con la mediana y escala los datos numéricos con escalado estándar (solo numericos)
pipeline_imputacionMediana_escaladoEstandar = ColumnTransformer(
    transformers=[
        ("imputación", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), NUMERICAS)
    ] + transformadores_categoricos
)


Para facilitar la inserción del modelo con el pipeline de preprocesado, se ha creado la siguiente función, la cual encapsula las etapas del preprocesado y el modelo. También se añade la posibilidad de proporcionar un diccionario de hiperparámetros para determinar la configuración óptima del modelo. Se trata de un grid de validación cruzada con 3 iteraciones y optimización basada en `f1_macro`. Se ha elegido esta métrica para optimizar frente a `recall_alto` (útil en este problema, ya que al ser uno de medicina en el que se mide el riesgo cardiovascular, no detectar un paciente de alto riesgo supondría el error más grave) porque `f1_macro` ofrece un equilibrio entre sensibilidad y precisión para todas las clases por igual obligando al modelo a no disparar las falsas alarmas, como sí ocurriría al usar `recall alto` como métrica, que buscaría clasificar a casi todos los pacientes como de alto riesgo en vez de discriminar entre pacientes, por lo que el modelo no estaría aprendiendo realmente.

In [16]:
# Para dado un pipeline y un modelo, construir un pipeline completo que incluya el preprocesado y el modelo (y opcionalmente un GridSearchCV si se proporciona un param_grid).
def construir_pipeline_evaluacion(pipeline_preprocesado, modelo, param_grid=None):
    pipeline_completo = Pipeline([
        ("preprocesado", pipeline_preprocesado),
        ("modelo", modelo)
    ])
    if param_grid is not None:
        return GridSearchCV(estimator=pipeline_completo, param_grid=param_grid, cv=3, scoring='f1_macro', n_jobs=-1)
    return pipeline_completo

- **`StratifiedShuffleSplit`** con 3 iteraciones y 30 % de test. Se estratifica porque las clases están desbalanceadas (la clase 2 tiene menos de la mitad de ejemplos que la 0).
- Para cada variante de preprocesado se generan las particiones estratificadas correspondientes a su tamaño muestral.
- **Métricas** de train y de test, para detectar sobreajuste:
  - *accuracy*;
  - *F1 macro*, que pesa igual las tres clases;
  - *recall de alto riesgo* (clase 2), la más crítica clínicamente: un paciente de alto riesgo no detectado es el peor error.

Cada métrica se muestra como **media ± desviación típica** de las 3 iteraciones. La desviación indica cuánto varía el resultado según la partición: si la diferencia entre dos preprocesados es menor que su desviación, no se puede asegurar que uno sea mejor que el otro.

`gap_f1` es la diferencia de F1 macro entre train y test en cada iteración: cuanto mayor, más sobreajuste.

In [17]:
N_SPLITS = 3
TEST_SIZE = 0.3

splitter = StratifiedShuffleSplit(n_splits=N_SPLITS, test_size=TEST_SIZE, random_state=SEED)

def metricas(y_real, y_pred, conjunto):
    return {
        f"{conjunto}_accuracy": accuracy_score(y_real, y_pred),
        f"{conjunto}_f1_macro": f1_score(y_real, y_pred, average="macro"),
        f"{conjunto}_recall_alto": recall_score(y_real, y_pred, labels=[2], average="macro"),
    }


def evaluar(pipeline, X, y, nombre_modelo, nombre_preprocesado, verbose=False):
    metricas_folds = {
        "train_accuracy": [], "train_f1_macro": [], "train_recall_alto": [],
        "test_accuracy": [], "test_f1_macro": [], "test_recall_alto": [],
        "gap_f1_macro": []
    }
    if verbose:
        print(f"Evaluando modelo {nombre_modelo} con preprocesado {nombre_preprocesado}...")
    for iteracion, (train_idx, test_idx) in enumerate(splitter.split(X, y), start=1):
        X_train_fold, X_test_fold = X.iloc[train_idx], X.iloc[test_idx]
        y_train_fold, y_test_fold = y.iloc[train_idx], y.iloc[test_idx]
        pipeline_fold = clone(pipeline)
        pipeline_fold.fit(X_train_fold, y_train_fold)
        y_pred_train = pipeline_fold.predict(X_train_fold)
        y_pred_test = pipeline_fold.predict(X_test_fold)
        m_train = metricas(y_train_fold, y_pred_train, "train")
        m_test = metricas(y_test_fold, y_pred_test, "test")
        gap_f1 = m_train['train_f1_macro'] - m_test['test_f1_macro']
        m_fold_extra = {"gap_f1_macro": gap_f1}
        for k, v in {**m_train, **m_test, **m_fold_extra}.items():
            metricas_folds[k].append(v)
        if verbose:
            print(f"--- Iteración {iteracion} ---")
            if hasattr(pipeline_fold, "best_params_"):
                print(f"  Mejores params -> {pipeline_fold.best_params_}")
            print(f"  F1 Macro   -> Train: {m_train['train_f1_macro']:.4f} | Test: {m_test['test_f1_macro']:.4f} | Gap: {gap_f1:.4f}")
            print(f"  Recall Alto -> Train: {m_train['train_recall_alto']:.4f} | Test: {m_test['test_recall_alto']:.4f}")
            print(f"  Accuracy   -> Train: {m_train['train_accuracy']:.4f} | Test: {m_test['test_accuracy']:.4f}")
    calculadas = {}
    for metrica, valores in metricas_folds.items():
        media = np.mean(valores)
        std = np.std(valores, ddof=1)
        calculadas[metrica] = f"{media:.4f} ± {std:.4f}"
    resumen_dict = {
        "preprocesado": nombre_preprocesado,
        "modelo": nombre_modelo,
        "train_f1_macro": calculadas["train_f1_macro"],
        "test_f1_macro": calculadas["test_f1_macro"],
        "gap_f1_macro": calculadas["gap_f1_macro"],
        "train_recall_alto": calculadas["train_recall_alto"],
        "test_recall_alto": calculadas["test_recall_alto"],
        "train_accuracy": calculadas["train_accuracy"],
        "test_accuracy": calculadas["test_accuracy"],
    }
    df_resumen = pd.DataFrame([resumen_dict])
    if verbose:
        print(f"\nResumen global:")
        print(f"  F1 MACRO   -> Train: {resumen_dict['train_f1_macro']} | Test: {resumen_dict['test_f1_macro']} | Gap: {resumen_dict['gap_f1_macro']}")
        print(f"  RECALL ALTO-> Train: {resumen_dict['train_recall_alto']} | Test: {resumen_dict['test_recall_alto']}")
        print(f"  ACCURACY   -> Train: {resumen_dict['train_accuracy']} | Test: {resumen_dict['test_accuracy']}")
    return df_resumen


Antes de comparar preprocesamientos de limpieza, primero se prueban los modelos para hacer un ajuste de hiperparámetos para no obtener resultados con overfitting o underfitting. Además, hay que decidir qué 2 modelos base se van a escoger. Se usa el preprocesamiento de limpieza P0 para hacer las pruebas de ajustes de hiperparámetros.

In [18]:
# Colesterol total ya se ha decidido eliminarse
if 'colesterol_total' in NUMERICAS:
    NUMERICAS.remove('colesterol_total')

X = DATOS["P0_base"].drop(columns=[TARGET])
y = DATOS["P0_base"][TARGET].astype(int)

In [19]:
# Logistic Regression
param_grid_lr = {
    'modelo__C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    'modelo__class_weight': [None, 'balanced']
}
pipeline_imputacionMediana_escaladoEstandar_lr = construir_pipeline_evaluacion(
    pipeline_imputacionMediana_escaladoEstandar,
    LogisticRegression(solver='lbfgs', max_iter=1000, random_state=SEED),
    param_grid_lr
)
res_imputacionMediana_escaladoEstandar_lr = evaluar(pipeline_imputacionMediana_escaladoEstandar_lr, X, y, "Logistic Regression", "P0_Base", True)

Evaluando modelo Logistic Regression con preprocesado P0_Base...
--- Iteración 1 ---
  Mejores params -> {'modelo__C': 1.0, 'modelo__class_weight': 'balanced'}
  F1 Macro   -> Train: 0.6819 | Test: 0.6494 | Gap: 0.0325
  Recall Alto -> Train: 0.7785 | Test: 0.7704
  Accuracy   -> Train: 0.7045 | Test: 0.6721
--- Iteración 2 ---
  Mejores params -> {'modelo__C': 100.0, 'modelo__class_weight': 'balanced'}
  F1 Macro   -> Train: 0.6714 | Test: 0.6750 | Gap: -0.0036
  Recall Alto -> Train: 0.7880 | Test: 0.7778
  Accuracy   -> Train: 0.6893 | Test: 0.6942
--- Iteración 3 ---
  Mejores params -> {'modelo__C': 0.1, 'modelo__class_weight': None}
  F1 Macro   -> Train: 0.6687 | Test: 0.6616 | Gap: 0.0071
  Recall Alto -> Train: 0.6994 | Test: 0.7333
  Accuracy   -> Train: 0.7045 | Test: 0.7046

Resumen global:
  F1 MACRO   -> Train: 0.6740 ± 0.0070 | Test: 0.6620 ± 0.0128 | Gap: 0.0120 ± 0.0186
  RECALL ALTO-> Train: 0.7553 ± 0.0486 | Test: 0.7605 ± 0.0238
  ACCURACY   -> Train: 0.6994 ± 0.008

Al ser un modelo computacionalmente eficiente, ya que se trata de un modelo linear, se prueba un rango de valores amplio para el hiperparámetro `C`. Se incluye un enfoque de balanceo de pesos automático frente a la distribución normal de las clases.

In [20]:
# Linear SVC
param_grid_svc = {
    'modelo__C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    'modelo__class_weight': [None, 'balanced']
}
pipeline_imputacionMediana_escaladoEstandar_svc = construir_pipeline_evaluacion(
    pipeline_imputacionMediana_escaladoEstandar,
    LinearSVC(dual='auto', max_iter=1000, random_state=SEED),
    param_grid_svc
)
res_imputacionMediana_escaladoEstandar_svc = evaluar(pipeline_imputacionMediana_escaladoEstandar_svc, X, y, "Linear SVC", "P0_Base", True)

Evaluando modelo Linear SVC con preprocesado P0_Base...
--- Iteración 1 ---
  Mejores params -> {'modelo__C': 10.0, 'modelo__class_weight': None}
  F1 Macro   -> Train: 0.6253 | Test: 0.5964 | Gap: 0.0289
  Recall Alto -> Train: 0.7595 | Test: 0.7630
  Accuracy   -> Train: 0.6918 | Test: 0.6736
--- Iteración 2 ---
  Mejores params -> {'modelo__C': 0.1, 'modelo__class_weight': None}
  F1 Macro   -> Train: 0.6073 | Test: 0.6186 | Gap: -0.0112
  Recall Alto -> Train: 0.7690 | Test: 0.7556
  Accuracy   -> Train: 0.6810 | Test: 0.6854
--- Iteración 3 ---
  Mejores params -> {'modelo__C': 0.1, 'modelo__class_weight': 'balanced'}
  F1 Macro   -> Train: 0.6347 | Test: 0.6033 | Gap: 0.0313
  Recall Alto -> Train: 0.8386 | Test: 0.8519
  Accuracy   -> Train: 0.6918 | Test: 0.6780

Resumen global:
  F1 MACRO   -> Train: 0.6224 ± 0.0139 | Test: 0.6061 ± 0.0113 | Gap: 0.0163 ± 0.0239
  RECALL ALTO-> Train: 0.7890 ± 0.0432 | Test: 0.7901 ± 0.0536
  ACCURACY   -> Train: 0.6882 ± 0.0062 | Test: 0.6790

Al igual que la regresión logística, LinearSVC es un clasificador lineal con un coste computacional muy bajo, además de que comparte los mismos hiperparámetros principales, lo que permite mantener exactamente el mismo espacio de búsqueda.

In [21]:
param_grid_svm = {
    'modelo__C': [1.0, 10.0, 100.0],
    'modelo__class_weight': [None, 'balanced'],
    'modelo__gamma': ['scale', 'auto']
}
pipeline_imputacionMediana_escaladoEstandar_svm = construir_pipeline_evaluacion(
    pipeline_imputacionMediana_escaladoEstandar,
    SVC(kernel='rbf', random_state=SEED),
    param_grid_svm
)
res_imputacionMediana_escaladoEstandar_svm = evaluar(pipeline_imputacionMediana_escaladoEstandar_svm, X, y, "SVM", "P0_Base", True)

Evaluando modelo SVM con preprocesado P0_Base...
--- Iteración 1 ---
  Mejores params -> {'modelo__C': 1.0, 'modelo__class_weight': 'balanced', 'modelo__gamma': 'auto'}
  F1 Macro   -> Train: 0.8056 | Test: 0.6481 | Gap: 0.1575
  Recall Alto -> Train: 0.9019 | Test: 0.7556
  Accuracy   -> Train: 0.8142 | Test: 0.6750
--- Iteración 2 ---
  Mejores params -> {'modelo__C': 1.0, 'modelo__class_weight': 'balanced', 'modelo__gamma': 'scale'}
  F1 Macro   -> Train: 0.8373 | Test: 0.6721 | Gap: 0.1651
  Recall Alto -> Train: 0.9272 | Test: 0.7556
  Accuracy   -> Train: 0.8427 | Test: 0.7001
--- Iteración 3 ---
  Mejores params -> {'modelo__C': 1.0, 'modelo__class_weight': 'balanced', 'modelo__gamma': 'scale'}
  F1 Macro   -> Train: 0.8412 | Test: 0.6402 | Gap: 0.2010
  Recall Alto -> Train: 0.9367 | Test: 0.7926
  Accuracy   -> Train: 0.8478 | Test: 0.6721

Resumen global:
  F1 MACRO   -> Train: 0.8280 ± 0.0195 | Test: 0.6535 ± 0.0166 | Gap: 0.1745 ± 0.0232
  RECALL ALTO-> Train: 0.9219 ± 0.01

In [ ]:
param_grid_svm = {
    'modelo__C': [0.001, 0.01, 0.1],
    'modelo__class_weight': [None, 'balanced'],
    'modelo__gamma': ['scale', 'auto']
}
pipeline_imputacionMediana_escaladoEstandar_svm = construir_pipeline_evaluacion(
    pipeline_imputacionMediana_escaladoEstandar,
    SVC(kernel='rbf', random_state=SEED),
    param_grid_svm
)
res_imputacionMediana_escaladoEstandar_svm = evaluar(pipeline_imputacionMediana_escaladoEstandar_svm, X, y, "SVM", "P0_Base", True)

Evaluando modelo SVM con preprocesado P0_Base...
--- Iteración 1 ---
  Mejores params -> {'modelo__C': 0.001, 'modelo__class_weight': None, 'modelo__gamma': 'scale'}
  F1 Macro   -> Train: 0.2221 | Test: 0.2224 | Gap: -0.0003
  Recall Alto -> Train: 0.0000 | Test: 0.0000
  Accuracy   -> Train: 0.4997 | Test: 0.5007
--- Iteración 2 ---
  Mejores params -> {'modelo__C': 0.001, 'modelo__class_weight': None, 'modelo__gamma': 'scale'}
  F1 Macro   -> Train: 0.2221 | Test: 0.2224 | Gap: -0.0003
  Recall Alto -> Train: 0.0000 | Test: 0.0000
  Accuracy   -> Train: 0.4997 | Test: 0.5007
--- Iteración 3 ---
  Mejores params -> {'modelo__C': 0.001, 'modelo__class_weight': None, 'modelo__gamma': 'scale'}
  F1 Macro   -> Train: 0.2221 | Test: 0.2224 | Gap: -0.0003
  Recall Alto -> Train: 0.0000 | Test: 0.0000
  Accuracy   -> Train: 0.4997 | Test: 0.5007

Resumen global:
  F1 MACRO   -> Train: 0.2221 ± 0.0000 | Test: 0.2224 ± 0.0000 | Gap: -0.0003 ± 0.0000
  RECALL ALTO-> Train: 0.0000 ± 0.0000 | Te

In [ ]:
param_grid_svm = {
    'modelo__C': [0.001, 0.01, 0.1],
    'modelo__class_weight': [None, 'balanced'],
    'modelo__gamma': ['scale', 'auto']
}
pipeline_imputacionMediana_escaladoEstandar_svm = construir_pipeline_evaluacion(
    pipeline_imputacionMediana_escaladoEstandar,
    SVC(kernel='linear', random_state=SEED),
    param_grid_svm
)
res_imputacionMediana_escaladoEstandar_svm = evaluar(pipeline_imputacionMediana_escaladoEstandar_svm, X, y, "SVM", "P0_Base", True)

Evaluando modelo SVM con preprocesado P0_Base...
--- Iteración 1 ---
  Mejores params -> {'modelo__C': 0.005, 'modelo__class_weight': 'balanced', 'modelo__gamma': 'scale'}
  F1 Macro   -> Train: 0.6598 | Test: 0.6451 | Gap: 0.0147
  Recall Alto -> Train: 0.7532 | Test: 0.8074
  Accuracy   -> Train: 0.6798 | Test: 0.6691
--- Iteración 2 ---
  Mejores params -> {'modelo__C': 0.005, 'modelo__class_weight': 'balanced', 'modelo__gamma': 'scale'}
  F1 Macro   -> Train: 0.6552 | Test: 0.6715 | Gap: -0.0163
  Recall Alto -> Train: 0.7880 | Test: 0.8000
  Accuracy   -> Train: 0.6734 | Test: 0.6942
--- Iteración 3 ---
  Mejores params -> {'modelo__C': 0.01, 'modelo__class_weight': 'balanced', 'modelo__gamma': 'scale'}
  F1 Macro   -> Train: 0.6589 | Test: 0.6531 | Gap: 0.0058
  Recall Alto -> Train: 0.7943 | Test: 0.7926
  Accuracy   -> Train: 0.6798 | Test: 0.6765

Resumen global:
  F1 MACRO   -> Train: 0.6580 ± 0.0025 | Test: 0.6566 ± 0.0135 | Gap: 0.0014 ± 0.0160
  RECALL ALTO-> Train: 0.7785

In [24]:
resultados_limpieza = pd.concat([res_imputacionMediana_escaladoEstandar_lr, res_imputacionMediana_escaladoEstandar_svc, res_imputacionMediana_escaladoEstandar_svm], ignore_index=True)

resultados_limpieza

,preprocesado,modelo,train_f1_macro,test_f1_macro,gap_f1_macro,train_recall_alto,test_recall_alto,train_accuracy,test_accuracy
0,P0_Base,Logistic Regression,0.6740 ± 0.0070,0.6620 ± 0.0128,0.0120 ± 0.0186,0.7553 ± 0.0486,0.7605 ± 0.0238,0.6994 ± 0.0088,0.6903 ± 0.0166
1,P0_Base,Linear SVC,0.6224 ± 0.0139,0.6061 ± 0.0113,0.0163 ± 0.0239,0.7890 ± 0.0432,0.7901 ± 0.0536,0.6882 ± 0.0062,0.6790 ± 0.0060
2,P0_Base,SVM,0.6580 ± 0.0025,0.6566 ± 0.0135,0.0014 ± 0.0160,0.7785 ± 0.0222,0.8000 ± 0.0074,0.6777 ± 0.0037,0.6800 ± 0.0129


In [27]:
for dato in DATOS.keys():
    if dato == "P0_base":
        continue  # Ya se evaluó P0_base antes del bucle
    X = DATOS[dato].drop(columns=[TARGET])
    y = DATOS[dato][TARGET].astype(int)
    res_lr = evaluar(pipeline_imputacionMediana_escaladoEstandar_lr, X, y, "Logistic Regression", dato, True)
    res_svc = evaluar(pipeline_imputacionMediana_escaladoEstandar_svc, X, y, "Linear SVC", dato, True)
    res_svm = evaluar(pipeline_imputacionMediana_escaladoEstandar_svm, X, y, "SVM", dato, True)
    resultados_limpieza = pd.concat([resultados_limpieza, res_lr, res_svc, res_svm], ignore_index=True)

Evaluando modelo Logistic Regression con preprocesado P1_limpieza...


--- Iteración 1 ---
  Mejores params -> {'modelo__C': 100.0, 'modelo__class_weight': 'balanced'}
  F1 Macro   -> Train: 0.6724 | Test: 0.6520 | Gap: 0.0204
  Recall Alto -> Train: 0.7911 | Test: 0.7704
  Accuracy   -> Train: 0.6940 | Test: 0.6756


--- Iteración 2 ---
  Mejores params -> {'modelo__C': 0.1, 'modelo__class_weight': None}
  F1 Macro   -> Train: 0.6619 | Test: 0.6581 | Gap: 0.0037
  Recall Alto -> Train: 0.6835 | Test: 0.7111
  Accuracy   -> Train: 0.7003 | Test: 0.6933


--- Iteración 3 ---
  Mejores params -> {'modelo__C': 1.0, 'modelo__class_weight': None}
  F1 Macro   -> Train: 0.6804 | Test: 0.6288 | Gap: 0.0516
  Recall Alto -> Train: 0.7184 | Test: 0.7037
  Accuracy   -> Train: 0.7098 | Test: 0.6711

Resumen global:
  F1 MACRO   -> Train: 0.6715 ± 0.0093 | Test: 0.6463 ± 0.0155 | Gap: 0.0252 ± 0.0243
  RECALL ALTO-> Train: 0.7310 ± 0.0549 | Test: 0.7284 ± 0.0365
  ACCURACY   -> Train: 0.7014 ± 0.0080 | Test: 0.6800 ± 0.0118
Evaluando modelo Linear SVC con preprocesado P1_limpieza...


--- Iteración 1 ---
  Mejores params -> {'modelo__C': 10.0, 'modelo__class_weight': 'balanced'}
  F1 Macro   -> Train: 0.6179 | Test: 0.5922 | Gap: 0.0257
  Recall Alto -> Train: 0.8228 | Test: 0.8148
  Accuracy   -> Train: 0.6794 | Test: 0.6607


--- Iteración 2 ---
  Mejores params -> {'modelo__C': 1.0, 'modelo__class_weight': None}
  F1 Macro   -> Train: 0.6104 | Test: 0.5868 | Gap: 0.0236
  Recall Alto -> Train: 0.7563 | Test: 0.7778
  Accuracy   -> Train: 0.6806 | Test: 0.6726


... (27 outputs left out)

--- Iteración 3 ---
  Mejores params -> {'modelo__C': 10.0, 'modelo__class_weight': 'balanced'}
  F1 Macro   -> Train: 0.5976 | Test: 0.6133 | Gap: -0.0157
  Recall Alto -> Train: 0.8307 | Test: 0.8444
  Accuracy   -> Train: 0.6722 | Test: 0.6741

Resumen global:
  F1 MACRO   -> Train: 0.6149 ± 0.0162 | Test: 0.5943 ± 0.0166 | Gap: 0.0206 ± 0.0316
  RECALL ALTO-> Train: 0.8083 ± 0.0334 | Test: 0.8049 ± 0.0373
  ACCURACY   -> Train: 0.6820 ± 0.0088 | Test: 0.6647 ± 0.0097
Evaluando modelo SVM con preprocesado P4_discretizacion...


--- Iteración 1 ---
  Mejores params -> {'modelo__C': 0.1, 'modelo__class_weight': 'balanced', 'modelo__gamma': 'scale'}
  F1 Macro   -> Train: 0.6633 | Test: 0.6328 | Gap: 0.0306
  Recall Alto -> Train: 0.7796 | Test: 0.7481
  Accuracy   -> Train: 0.6818 | Test: 0.6562


--- Iteración 2 ---
  Mejores params -> {'modelo__C': 0.01, 'modelo__class_weight': 'balanced', 'modelo__gamma': 'scale'}
  F1 Macro   -> Train: 0.6561 | Test: 0.6432 | Gap: 0.0128
  Recall Alto -> Train: 0.7891 | Test: 0.7630
  Accuracy   -> Train: 0.6786 | Test: 0.6622


--- Iteración 3 ---
  Mejores params -> {'modelo__C': 0.01, 'modelo__class_weight': 'balanced', 'modelo__gamma': 'scale'}
  F1 Macro   -> Train: 0.6386 | Test: 0.6626 | Gap: -0.0240
  Recall Alto -> Train: 0.7764 | Test: 0.8222
  Accuracy   -> Train: 0.6633 | Test: 0.6786

Resumen global:
  F1 MACRO   -> Train: 0.6526 ± 0.0127 | Test: 0.6462 ± 0.0151 | Gap: 0.0065 ± 0.0278
  RECALL ALTO-> Train: 0.7817 ± 0.0067 | Test: 0.7778 ± 0.0392
  ACCURACY   -> Train: 0.6745 ± 0.0099 | Test: 0.6657 ± 0.0116


In [28]:
resultados_limpieza

,preprocesado,modelo,train_f1_macro,test_f1_macro,gap_f1_macro,train_recall_alto,test_recall_alto,train_accuracy,test_accuracy
0,P0_Base,Logistic Regression,0.6740 ± 0.0070,0.6620 ± 0.0128,0.0120 ± 0.0186,0.7553 ± 0.0486,0.7605 ± 0.0238,0.6994 ± 0.0088,0.6903 ± 0.0166
1,P0_Base,Linear SVC,0.6224 ± 0.0139,0.6061 ± 0.0113,0.0163 ± 0.0239,0.7890 ± 0.0432,0.7901 ± 0.0536,0.6882 ± 0.0062,0.6790 ± 0.0060
2,P0_Base,SVM,0.6766 ± 0.0072,0.6619 ± 0.0151,0.0147 ± 0.0223,0.7954 ± 0.0128,0.7901 ± 0.0154,0.6965 ± 0.0077,0.6844 ± 0.0137
3,P1_limpieza,Logistic Regression,0.6715 ± 0.0093,0.6463 ± 0.0155,0.0252 ± 0.0243,0.7310 ± 0.0549,0.7284 ± 0.0365,0.7014 ± 0.0080,0.6800 ± 0.0118
4,P1_limpieza,Linear SVC,0.6161 ± 0.0051,0.5826 ± 0.0123,0.0335 ± 0.0154,0.7827 ± 0.0353,0.7778 ± 0.0370,0.6825 ± 0.0044,0.6642 ± 0.0073
5,P1_limpieza,SVM,0.6665 ± 0.0134,0.6451 ± 0.0156,0.0214 ± 0.0183,0.7278 ± 0.0560,0.7259 ± 0.0412,0.6974 ± 0.0080,0.6800 ± 0.0053
6,P2_diabetes,Logistic Regression,0.6676 ± 0.0145,0.6497 ± 0.0166,0.0179 ± 0.0311,0.7605 ± 0.0640,0.7358 ± 0.0334,0.6921 ± 0.0044,0.6765 ± 0.0224
7,P2_diabetes,Linear SVC,0.6059 ± 0.0100,0.5839 ± 0.0164,0.0220 ± 0.0214,0.8143 ± 0.0341,0.7951 ± 0.0408,0.6753 ± 0.0072,0.6602 ± 0.0048
8,P2_diabetes,SVM,0.6674 ± 0.0100,0.6423 ± 0.0133,0.0251 ± 0.0195,0.7289 ± 0.0557,0.7160 ± 0.0334,0.6980 ± 0.0065,0.6770 ± 0.0128
9,P3_pcr,Logistic Regression,0.6626 ± 0.0064,0.6500 ± 0.0205,0.0126 ± 0.0233,0.7732 ± 0.0096,0.7580 ± 0.0434,0.6822 ± 0.0057,0.6677 ± 0.0135


P0 > P3 > P2 > P1, pero P0 tiene valores erroneos o cientificamente imposibles, por lo que no se puede usar. Parece que P3 es el que se va a elegir.

**P4** ya se ha evaluado en el bucle anterior junto con P1, P2 y P3, porque es un preprocesado más de `PREPROCESADOS`. Para evaluar **P5, P6 y P7** con los mismos tres modelos y las mismas rejillas
se reutilizan los `GridSearchCV` ya definidos (`pipeline_..._lr`, `_svc` y `_svm`) y solo se inserta entre el preprocesado y el modelo el paso que corresponda (selección de variables y/o sobremuestreo).
`construir_pipeline_evaluacion_extra` hace lo mismo que `construir_pipeline_evaluacion`, pero con un `Pipeline` de `imbalanced-learn`, que permite meter un sobremuestreador que **solo actúa al ajustar
y nunca sobre el test**. Dentro del `GridSearchCV`, el sobremuestreo se aplica únicamente al train de cada partición interna.

In [29]:
from imblearn.pipeline import Pipeline as ImbPipeline


def construir_pipeline_evaluacion_extra(busqueda, sampler=None, seleccion=None):
    # busqueda: un GridSearchCV como los de arriba, con un pipeline [preprocesado, modelo]
    pasos = [(nombre, clone(paso)) for nombre, paso in busqueda.estimator.steps]
    extra = []
    if seleccion is not None:
        extra.append(("seleccion", clone(seleccion)))
    if sampler is not None:
        extra.append(("sampler", clone(sampler)))
    pasos = pasos[:-1] + extra + pasos[-1:]
    return GridSearchCV(estimator=ImbPipeline(pasos), param_grid=busqueda.param_grid, cv=3, scoring='f1_macro', n_jobs=-1)


MODELOS_EVALUACION = {
    "Logistic Regression": pipeline_imputacionMediana_escaladoEstandar_lr,
    "Linear SVC": pipeline_imputacionMediana_escaladoEstandar_svc,
    "SVM": pipeline_imputacionMediana_escaladoEstandar_svm,
}

resultados_extra = []
for nombre, config in PREPROCESADOS_PIPELINE.items():
    X = DATOS[config["base"]].drop(columns=[TARGET])
    y = DATOS[config["base"]][TARGET].astype(int)
    for nombre_modelo, busqueda in MODELOS_EVALUACION.items():
        pipeline = construir_pipeline_evaluacion_extra(busqueda, config["sampler"], config["seleccion"])
        resultados_extra.append(evaluar(pipeline, X, y, nombre_modelo, nombre, True))

resultados_preprocesados = pd.concat([resultados_limpieza] + resultados_extra, ignore_index=True)

Evaluando modelo Logistic Regression con preprocesado P5_smote...


--- Iteración 1 ---
  Mejores params -> {'modelo__C': 0.1, 'modelo__class_weight': None}
  F1 Macro   -> Train: 0.6629 | Test: 0.6309 | Gap: 0.0320
  Recall Alto -> Train: 0.7732 | Test: 0.7407
  Accuracy   -> Train: 0.6843 | Test: 0.6533


--- Iteración 2 ---
  Mejores params -> {'modelo__C': 0.1, 'modelo__class_weight': None}
  F1 Macro   -> Train: 0.6661 | Test: 0.6485 | Gap: 0.0176
  Recall Alto -> Train: 0.7859 | Test: 0.7481
  Accuracy   -> Train: 0.6881 | Test: 0.6726


--- Iteración 3 ---
  Mejores params -> {'modelo__C': 0.1, 'modelo__class_weight': None}
  F1 Macro   -> Train: 0.6493 | Test: 0.6541 | Gap: -0.0048
  Recall Alto -> Train: 0.7668 | Test: 0.8000
  Accuracy   -> Train: 0.6709 | Test: 0.6696

Resumen global:
  F1 MACRO   -> Train: 0.6594 ± 0.0089 | Test: 0.6445 ± 0.0121 | Gap: 0.0149 ± 0.0186
  RECALL ALTO-> Train: 0.7753 ± 0.0098 | Test: 0.7630 ± 0.0323
  ACCURACY   -> Train: 0.6811 ± 0.0090 | Test: 0.6652 ± 0.0104
Evaluando modelo Linear SVC con preprocesado P5_smote...


--- Iteración 1 ---
  Mejores params -> {'modelo__C': 0.1, 'modelo__class_weight': None}
  F1 Macro   -> Train: 0.6224 | Test: 0.5819 | Gap: 0.0405
  Recall Alto -> Train: 0.8307 | Test: 0.8074
  Accuracy   -> Train: 0.6716 | Test: 0.6369


--- Iteración 2 ---
  Mejores params -> {'modelo__C': 0.1, 'modelo__class_weight': None}
  F1 Macro   -> Train: 0.6331 | Test: 0.6116 | Gap: 0.0215
  Recall Alto -> Train: 0.8658 | Test: 0.8222
  Accuracy   -> Train: 0.6849 | Test: 0.6652


... (18 outputs left out)

--- Iteración 3 ---
  Mejores params -> {'modelo__C': 1.0, 'modelo__class_weight': 'balanced'}
  F1 Macro   -> Train: 0.5952 | Test: 0.6081 | Gap: -0.0129
  Recall Alto -> Train: 0.8211 | Test: 0.8222
  Accuracy   -> Train: 0.6690 | Test: 0.6726

Resumen global:
  F1 MACRO   -> Train: 0.6095 ± 0.0124 | Test: 0.6014 ± 0.0105 | Gap: 0.0080 ± 0.0200
  RECALL ALTO-> Train: 0.8296 ± 0.0080 | Test: 0.8000 ± 0.0196
  ACCURACY   -> Train: 0.6758 ± 0.0067 | Test: 0.6662 ± 0.0099
Evaluando modelo SVM con preprocesado P7_rfe...


--- Iteración 1 ---
  Mejores params -> {'modelo__C': 0.1, 'modelo__class_weight': 'balanced', 'modelo__gamma': 'scale'}
  F1 Macro   -> Train: 0.6496 | Test: 0.6296 | Gap: 0.0201
  Recall Alto -> Train: 0.7732 | Test: 0.7407
  Accuracy   -> Train: 0.6690 | Test: 0.6533


--- Iteración 2 ---
  Mejores params -> {'modelo__C': 0.01, 'modelo__class_weight': 'balanced', 'modelo__gamma': 'scale'}
  F1 Macro   -> Train: 0.6479 | Test: 0.6393 | Gap: 0.0085
  Recall Alto -> Train: 0.7796 | Test: 0.7407
  Accuracy   -> Train: 0.6690 | Test: 0.6607


--- Iteración 3 ---
  Mejores params -> {'modelo__C': 0.1, 'modelo__class_weight': 'balanced', 'modelo__gamma': 'scale'}
  F1 Macro   -> Train: 0.6522 | Test: 0.6575 | Gap: -0.0053
  Recall Alto -> Train: 0.7955 | Test: 0.8000
  Accuracy   -> Train: 0.6747 | Test: 0.6711

Resumen global:
  F1 MACRO   -> Train: 0.6499 ± 0.0022 | Test: 0.6421 ± 0.0142 | Gap: 0.0078 ± 0.0127
  RECALL ALTO-> Train: 0.7827 ± 0.0115 | Test: 0.7605 ± 0.0342
  ACCURACY   -> Train: 0.6709 ± 0.0033 | Test: 0.6617 ± 0.0090


Tabla con **todos los preprocesados (P0 a P7) y los tres modelos**, y la media de los tres modelos para cada preprocesado. Los preprocesados con filas eliminadas por PCR (P3 a P7) comparten partición entre sí.

In [30]:
def media_metrica(resultados, preprocesado, metrica):
    # Las métricas están guardadas como texto "media ± desviación"; se toma la media
    sub = resultados[resultados["preprocesado"] == preprocesado]
    return sub[metrica].str.split(" ± ").str[0].astype(float).mean()


display(resultados_preprocesados)

resumen_preprocesados = pd.DataFrame({
    "f1_macro_test_medio": {p: media_metrica(resultados_preprocesados, p, "test_f1_macro") for p in resultados_preprocesados["preprocesado"].unique()},
    "recall_alto_test_medio": {p: media_metrica(resultados_preprocesados, p, "test_recall_alto") for p in resultados_preprocesados["preprocesado"].unique()},
    "accuracy_test_medio": {p: media_metrica(resultados_preprocesados, p, "test_accuracy") for p in resultados_preprocesados["preprocesado"].unique()},
}).round(4)
resumen_preprocesados

,preprocesado,modelo,train_f1_macro,test_f1_macro,gap_f1_macro,train_recall_alto,test_recall_alto,train_accuracy,test_accuracy
0,P0_Base,Logistic Regression,0.6740 ± 0.0070,0.6620 ± 0.0128,0.0120 ± 0.0186,0.7553 ± 0.0486,0.7605 ± 0.0238,0.6994 ± 0.0088,0.6903 ± 0.0166
1,P0_Base,Linear SVC,0.6224 ± 0.0139,0.6061 ± 0.0113,0.0163 ± 0.0239,0.7890 ± 0.0432,0.7901 ± 0.0536,0.6882 ± 0.0062,0.6790 ± 0.0060
2,P0_Base,SVM,0.6766 ± 0.0072,0.6619 ± 0.0151,0.0147 ± 0.0223,0.7954 ± 0.0128,0.7901 ± 0.0154,0.6965 ± 0.0077,0.6844 ± 0.0137
3,P1_limpieza,Logistic Regression,0.6715 ± 0.0093,0.6463 ± 0.0155,0.0252 ± 0.0243,0.7310 ± 0.0549,0.7284 ± 0.0365,0.7014 ± 0.0080,0.6800 ± 0.0118
4,P1_limpieza,Linear SVC,0.6161 ± 0.0051,0.5826 ± 0.0123,0.0335 ± 0.0154,0.7827 ± 0.0353,0.7778 ± 0.0370,0.6825 ± 0.0044,0.6642 ± 0.0073
5,P1_limpieza,SVM,0.6665 ± 0.0134,0.6451 ± 0.0156,0.0214 ± 0.0183,0.7278 ± 0.0560,0.7259 ± 0.0412,0.6974 ± 0.0080,0.6800 ± 0.0053
6,P2_diabetes,Logistic Regression,0.6676 ± 0.0145,0.6497 ± 0.0166,0.0179 ± 0.0311,0.7605 ± 0.0640,0.7358 ± 0.0334,0.6921 ± 0.0044,0.6765 ± 0.0224
7,P2_diabetes,Linear SVC,0.6059 ± 0.0100,0.5839 ± 0.0164,0.0220 ± 0.0214,0.8143 ± 0.0341,0.7951 ± 0.0408,0.6753 ± 0.0072,0.6602 ± 0.0048
8,P2_diabetes,SVM,0.6674 ± 0.0100,0.6423 ± 0.0133,0.0251 ± 0.0195,0.7289 ± 0.0557,0.7160 ± 0.0334,0.6980 ± 0.0065,0.6770 ± 0.0128
9,P3_pcr,Logistic Regression,0.6626 ± 0.0064,0.6500 ± 0.0205,0.0126 ± 0.0233,0.7732 ± 0.0096,0.7580 ± 0.0434,0.6822 ± 0.0057,0.6677 ± 0.0135


,f1_macro_test_medio,recall_alto_test_medio,accuracy_test_medio
P0_Base,0.6433,0.7802,0.6846
P1_limpieza,0.6247,0.7440,0.6747
P2_diabetes,0.6253,0.7490,0.6712
P3_pcr,0.6265,0.7679,0.6650
P4_discretizacion,0.6317,0.7769,0.6693
P5_smote,0.6297,0.7860,0.6609
P6_adasyn,0.6268,0.8041,0.6558
P7_rfe,0.6308,0.7761,0.6647


Para saber **qué variables elimina RFE y por qué** se ajusta una vez sobre todos los datos de P4. Esta celda es solo informativa: las métricas de arriba se calculan con la selección hecha dentro de cada partición de train.
Se muestra la puntuación de validación (F1 macro) según el número de variables, la mejor, el umbral de un error estándar y las variables que se eliminan.

In [31]:
X = DATOS["P4_discretizacion"].drop(columns=[TARGET])
y = DATOS["P4_discretizacion"][TARGET].astype(int)

preprocesador_info = clone(pipeline_imputacionMediana_escaladoEstandar)
seleccion_info = clone(PREPROCESADOS_PIPELINE["P7_rfe"]["seleccion"])
seleccion_info.fit(preprocesador_info.fit_transform(X, y), y)

nombres = preprocesador_info.get_feature_names_out()
media = seleccion_info.cv_results_["mean_test_score"]
n_particiones = sum(k.startswith("split") and k.endswith("_test_score") for k in seleccion_info.cv_results_)
error = seleccion_info.cv_results_["std_test_score"] / np.sqrt(n_particiones)
n_variables = np.arange(seleccion_info.min_features_to_select, seleccion_info.min_features_to_select + len(media))
print(pd.DataFrame({"n_variables": n_variables, "f1_macro_validacion": media.round(4)}).iloc[::3].to_string(index=False))
print(f"\nMejor puntuación: {media.max():.4f} con {n_variables[media.argmax()]} variables (error estándar {error[media.argmax()]:.4f}).")
print(f"Umbral (mejor - 1 error estándar): {media.max() - error[media.argmax()]:.4f}")
print(f"RFE conserva {seleccion_info.n_features_} de {len(nombres)} variables, el menor número que alcanza el umbral.")
print("Eliminadas:", ", ".join(n for n, s in zip(nombres, seleccion_info.support_) if not s) or "ninguna")

 n_variables  f1_macro_validacion
           5               0.4930
           8               0.5375
          11               0.5615
          14               0.5910
          17               0.6070
          20               0.6145
          23               0.6366
          26               0.6334
          29               0.6462

Mejor puntuación: 0.6473 con 30 variables (error estándar 0.0104).
Umbral (mejor - 1 error estándar): 0.6369
RFE conserva 27 de 30 variables, el menor número que alcanza el umbral.
Eliminadas: imputación__trigliceridos, imputación__frecuencia_cardiaca_reposo, nominales__antecedentes_familiares_tipo_Paterno


**P4 frente a P3.** P4 mejora ligeramente a P3: el F1 macro de test medio de los tres modelos pasa de 0,6265 a 0,6317, el recall de alto riesgo de 0,7679 a 0,7769 y el `gap_f1_macro` baja en los tres modelos. La mejora de F1 por modelo (entre +0,005 y +0,006) es menor que la desviación típica entre particiones (0,015-0,021), así que no se puede asegurar que sea mejor por las métricas solas. Como no empeora el F1 macro en ningún modelo y es coherente con el EDA, P4 es la base de P5, P6 y P7. La excepción es el recall de alto riesgo de la regresión logística, que baja de 0,7580 a 0,7481 (dentro de su desviación, 0,07).

**Desbalanceo y RFE (sobre P4).**

- **SMOTE y ADASYN** no suben el F1 macro (0,6297 y 0,6268 frente a 0,6317 de P4), pero **sí suben el recall de alto riesgo**, la métrica más importante clínicamente: de 0,7769 en P4 a 0,7860 con SMOTE y 0,8041 con ADASYN. El efecto se nota sobre todo en `LinearSVC` (0,8049 en P4, 0,8321 con SMOTE y 0,8370 con ADASYN), que es también el modelo con peor F1. El precio es algo menos de accuracy en test (entre 0,8 y 1,4 puntos).
- **RFE** con la regla del error estándar elimina 3 de las 30 variables (`trigliceridos`, `frecuencia_cardiaca_reposo` y la dummy `antecedentes_familiares_tipo_Paterno`) y **mantiene el rendimiento de P4**: F1 macro 0,6308 frente a 0,6317 y recall de alto riesgo 0,7761 frente a 0,7769, con un `gap_f1_macro` algo menor en `LinearSVC` (0,0080 frente a 0,0206). El `RFECV` sin esa regla conservaría las 30 variables, porque la puntuación de validación sube con cada variable que se añade (de 0,493 con 5 a 0,647 con 30). La selección, por tanto, simplifica el modelo sin mejorarlo. Al quitar la dummy `Paterno`, esos pacientes pasan a tratarse como los de la categoría de referencia de `antecedentes_familiares_tipo`.

La diferencia de F1 entre P4, P5, P6 y P7 está dentro del ruido, así que la elección depende del objetivo: si se prioriza detectar a los pacientes de alto riesgo, ADASYN (o SMOTE) sobre P4; si se prefiere el modelo más sencillo, P4 solo o con RFE (27 variables). P0 tiene el F1 más alto, pero conserva valores erróneos o imposibles (ver conclusión anterior), por lo que no se usa como referencia. Estas comparaciones se hacen con una rejilla mínima de hiperparámetros: el ajuste completo es parte del modelado.